# M06 · RAG avançado: prática

Vamos partir de um RAG simples sobre legislação portuguesa (Código do Trabalho e Código do IVA) e acrescentar técnicas uma a uma, medindo o ganho de cada uma:

1. Linha de base: pesquisa vetorial sobre chunks de tamanho fixo.
2. Unidades com significado: artigos como pais, números como filhos.
3. Pesquisa híbrida com RRF.
4. Reranking com cross-encoder.
5. Transformar a pergunta: multi-query e HyDE.
6. Contextual retrieval.
7. Grafo de referências entre artigos.
8. Agentic RAG: o modelo decide o que pesquisar.

**Preparação dos textos.** Em [diariodarepublica.pt](https://diariodarepublica.pt), procura «Código do Trabalho» e «Código do IVA» em *Legislação Consolidada* e descarrega as versões em PDF para `use-case/legislacao/` com os nomes `CT.pdf` e `CIVA.pdf`.

**Requisitos:** `uv sync --extra rag`, `ANTHROPIC_API_KEY`. O reranker (cerca de 2 GB) corre em CPU, mas é mais rápido com GPU.

Lição de apoio: [licao.html](licao.html)

In [ ]:
import json, os, re, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from anthropic import Anthropic
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

sys.path.insert(0, "use-case")
from legislacao import texto_pdf, partir_artigos, partir_numeros, tokenizar, rrf, NOMES

load_dotenv()
llm = Anthropic()
MODELO = os.getenv("MODELO", "claude-haiku-5-5")
emb = SentenceTransformer("intfloat/multilingual-e5-base")

PASTA = Path("use-case/legislacao")
textos = {f.stem.upper(): texto_pdf(f) for f in sorted(PASTA.glob("*.pdf"))}
print({k: f"{len(v):,} caracteres" for k, v in textos.items()})

testes = pd.read_json("use-case/perguntas_teste.jsonl", lines=True, dtype={"artigo": str})
testes["alvo"] = testes.diploma + ":" + testes.artigo
testes

As perguntas de teste foram escritas como um trabalhador perguntaria, sem o vocabulário jurídico. Confirma os artigos de referência na versão que descarregaste: a numeração pode mudar com alterações legislativas.

## 1. Linha de base: chunks de tamanho fixo

In [ ]:
def chunks_fixos(texto, diploma, tam=1000, sobre=150):
    out = []
    for i in range(0, len(texto), tam - sobre):
        pedaco = texto[i:i + tam]
        # artigo a que o chunk pertence: último "Artigo N.º" antes do meio do chunk
        ant = list(re.finditer(r"Artigo\s+(\d+)\.?\s*º", texto[:i + tam // 2]))
        out.append({"texto": pedaco, "alvo": f"{diploma}:{ant[-1].group(1)}" if ant else None})
    return out

base = [c for d, t in textos.items() for c in chunks_fixos(t, d)]
Vb = emb.encode(["passage: " + c["texto"] for c in base], normalize_embeddings=True, batch_size=32, show_progress_bar=True)

def avaliar(fn_pesquisa, nome, k=5):
    """fn_pesquisa(pergunta) -> lista de chaves 'DIPLOMA:N' ordenadas, sem repetidos."""
    t0 = time.time()
    res = [fn_pesquisa(p) for p in testes.pergunta]
    r = {"técnica": nome,
         "top1": np.mean([x[:1] == [a] for x, a in zip(res, testes.alvo)]),
         f"top{k}": np.mean([a in x[:k] for x, a in zip(res, testes.alvo)]),
         "s/pergunta": (time.time() - t0) / len(testes)}
    return {k2: round(v, 2) if isinstance(v, float) else v for k2, v in r.items()}

def unicos(seq):
    vistos = []
    for s in seq:
        if s and s not in vistos:
            vistos.append(s)
    return vistos

def pesquisa_base(p, k=5):
    q = emb.encode("query: " + p, normalize_embeddings=True)
    return unicos(base[i]["alvo"] for i in np.argsort(-(Vb @ q))[:30])[:k]

tabela = [avaliar(pesquisa_base, "1. vetorial, chunks fixos")]
pd.DataFrame(tabela)

## 2. Artigos (pais) e números (filhos)

In [ ]:
artigos = [a for d, t in textos.items() for a in partir_artigos(t, d)]
filhos = [(i, t) for i, a in enumerate(artigos) for t in partir_numeros(a)]
print(len(artigos), "artigos |", len(filhos), "filhos")
print(artigos[0].cabecalho(), "\n", artigos[0].texto[:300])

V = emb.encode([f"passage: {artigos[p].cabecalho()}\n{t}" for p, t in filhos], normalize_embeddings=True, batch_size=32, show_progress_bar=True)

def pais_de(indices, k=5):
    return unicos(artigos[filhos[i][0]].chave for i in indices)[:k]

def vetorial(p, n=50):
    q = emb.encode("query: " + p, normalize_embeddings=True)
    return list(np.argsort(-(V @ q))[:n])

tabela.append(avaliar(lambda p: pais_de(vetorial(p)), "2. vetorial, artigo/número + cabeçalho"))
pd.DataFrame(tabela)

## 3. Pesquisa híbrida

In [ ]:
bm25 = BM25Okapi([tokenizar(artigos[p].cabecalho() + " " + t) for p, t in filhos])

def lexical(p, n=50):
    return list(np.argsort(-bm25.get_scores(tokenizar(p)))[:n])

def hibrida(p, n=50):
    return rrf([vetorial(p, n), lexical(p, n)])[:n]

tabela.append(avaliar(lambda p: pais_de(lexical(p)), "3a. BM25"))
tabela.append(avaliar(lambda p: pais_de(hibrida(p)), "3b. híbrida (RRF)"))
pd.DataFrame(tabela)

## 4. Reranking

In [ ]:
rr = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512)

def rerank(p, cand):
    s = rr.predict([(p, artigos[filhos[c][0]].cabecalho() + "\n" + filhos[c][1]) for c in cand])
    return [c for _, c in sorted(zip(s, cand), key=lambda x: -x[0])]

tabela.append(avaliar(lambda p: pais_de(rerank(p, hibrida(p, 50))), "4. híbrida + reranking"))
pd.DataFrame(tabela)

**Exercício 4.1.** Varia o número de candidatos enviados ao reranker (10, 30, 50, 100). Desenha recall@5 e tempo por pergunta em função desse número. Onde está o ponto ótimo?

## 5. Transformar a pergunta

In [ ]:
def gerar(prompt, max_tokens=300):
    return llm.messages.create(model=MODELO, max_tokens=max_tokens, temperature=0,
                               messages=[{"role": "user", "content": prompt}]).content[0].text

def multi_query(p, n=3):
    t = gerar(f"Reescreve a pergunta de {n} formas diferentes usando a terminologia jurídica da legislação laboral e fiscal portuguesa. "
              f"Uma por linha, sem numeração nem comentários.\n\nPergunta: {p}")
    return [p] + [l.strip("-• ").strip() for l in t.splitlines() if l.strip()][:n]

def hyde(p):
    return gerar(f"Escreve um excerto curto (3 a 5 frases) de um artigo da legislação portuguesa que responda à pergunta, "
                 f"no estilo de um código legal (números, «O trabalhador tem direito...»). Não precisa de ser exato.\n\nPergunta: {p}")

print(multi_query(testes.pergunta[6]))
print(hyde(testes.pergunta[6]))

In [ ]:
def pesquisa_multi(p):
    qs = multi_query(p)
    cand = rrf([hibrida(q, 30) for q in qs])[:50]
    return pais_de(rerank(p, cand))

def pesquisa_hyde(p):
    h = hyde(p)
    cand = rrf([hibrida(p, 50), vetorial(h, 50)])[:50]
    return pais_de(rerank(p, cand))

tabela.append(avaliar(pesquisa_multi, "5a. multi-query + híbrida + rerank"))
tabela.append(avaliar(pesquisa_hyde, "5b. HyDE + híbrida + rerank"))
pd.DataFrame(tabela)

## 6. Contextual retrieval

Geramos uma frase de contexto para cada filho a partir do artigo completo e do título do capítulo. Para poupar custos, aplicamos só aos artigos que aparecem nos candidatos das perguntas de teste (num projeto real, a todo o corpus, uma única vez e com prompt caching).

In [ ]:
def contexto_para(p_idx, texto_filho):
    a = artigos[p_idx]
    return gerar(f"<artigo>\n{a.cabecalho()}\n{a.texto[:3000]}\n</artigo>\n\n<excerto>\n{texto_filho}\n</excerto>\n\n"
                 "Escreve uma única frase que situe o excerto no artigo e no diploma e diga de que trata, em linguagem simples, "
                 "incluindo sinónimos comuns (por exemplo «horas extra» para trabalho suplementar). Responde só com a frase.", 120)

relevantes = sorted({c for p in testes.pergunta for c in hibrida(p, 30)})
print(len(relevantes), "filhos a contextualizar")
contextos = {}
for c in relevantes:
    contextos[c] = contexto_para(*filhos[c])

V_ctx = V.copy()
idx = list(contextos)
V_ctx[idx] = emb.encode([f"passage: {contextos[c]}\n{artigos[filhos[c][0]].cabecalho()}\n{filhos[c][1]}" for c in idx], normalize_embeddings=True)

def vetorial_ctx(p, n=50):
    q = emb.encode("query: " + p, normalize_embeddings=True)
    return list(np.argsort(-(V_ctx @ q))[:n])

tabela.append(avaliar(lambda p: pais_de(rerank(p, rrf([vetorial_ctx(p), lexical(p)])[:50])), "6. contextual + híbrida + rerank"))
pd.DataFrame(tabela)

## 7. Grafo de referências

Os artigos citam-se uns aos outros. Construímos o grafo com `networkx` e, depois da pesquisa, acrescentamos os artigos referidos pelos primeiros resultados. Isto não melhora o top 1 (é uma expansão), mas dá ao LLM o contexto de que precisa para perguntas que dependem de vários artigos.

In [ ]:
import networkx as nx

G = nx.DiGraph()
chave_idx = {a.chave: i for i, a in enumerate(artigos)}
for a in artigos:
    for r in a.refs:
        if f"{a.diploma}:{r}" in chave_idx:
            G.add_edge(a.chave, f"{a.diploma}:{r}")
print(G.number_of_nodes(), "nós,", G.number_of_edges(), "referências")
print("Mais citados:", sorted(G.in_degree, key=lambda x: -x[1])[:10])

def expandir(chaves, por_artigo=2, max_extra=3):
    extra = []
    for c in chaves[:2]:
        for viz in list(G.successors(c))[:por_artigo] if c in G else []:
            if viz not in chaves and viz not in extra:
                extra.append(viz)
    return chaves + extra[:max_extra]

expandir(pais_de(rerank(testes.pergunta[3], hibrida(testes.pergunta[3]))))

## 8. Agentic RAG

Damos ao modelo uma ferramenta de pesquisa e deixamo-lo decidir quantas pesquisas fazer. É a primeira versão de um agente; no módulo 8 vamos aprofundar.

In [ ]:
def melhor_pesquisa(consulta, k=4):
    return pais_de(rerank(consulta, hibrida(consulta)), k)

FERRAMENTAS = [{
    "name": "pesquisar_legislacao",
    "description": "Pesquisa artigos do Código do Trabalho (CT) e do Código do IVA (CIVA). Devolve os artigos mais relevantes com o texto. "
                   "Usa termos jurídicos; podes chamar várias vezes com consultas diferentes.",
    "input_schema": {"type": "object", "properties": {"consulta": {"type": "string"}}, "required": ["consulta"]},
}]
SYSTEM_AGENTE = ("És um assistente de legislação portuguesa. Pesquisa até teres os artigos necessários e responde citando "
                 "diploma e artigo. Se não encontrares, diz que não encontraste.")

def agente(pergunta, max_passos=5):
    msgs = [{"role": "user", "content": pergunta}]
    for passo in range(max_passos):
        r = llm.messages.create(model=MODELO, max_tokens=1000, system=SYSTEM_AGENTE, tools=FERRAMENTAS, messages=msgs)
        msgs.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use":
            return next(b.text for b in r.content if b.type == "text"), passo
        resultados = []
        for b in r.content:
            if b.type == "tool_use":
                chaves = melhor_pesquisa(b.input["consulta"])
                print(f"  🔎 {b.input['consulta']!r} → {chaves}")
                txt = "\n\n".join(f"[{c}] {artigos[chave_idx[c]].cabecalho()}\n{artigos[chave_idx[c]].texto[:1500]}" for c in chaves)
                resultados.append({"type": "tool_result", "tool_use_id": b.id, "content": txt})
        msgs.append({"role": "user", "content": resultados})
    return "Limite de passos atingido.", max_passos

resposta, passos = agente("Um trabalhador com contrato há 8 meses quer sair. Que aviso prévio tem de dar e tem direito a subsídio de férias proporcional?")
print(f"\n({passos} pesquisas)\n{resposta}")

## Resultados

In [ ]:
pd.DataFrame(tabela).set_index("técnica")

**Exercício final.** Com 13 perguntas, cada pergunta vale 7,7 pontos percentuais: escreve mais 20 perguntas de teste (podes pedir a um LLM que as gere a partir de artigos escolhidos ao acaso e rever) e repete a tabela. Que técnicas compensam o custo e a latência?

## Use case: assistente de legislação portuguesa

`use-case/legislacao.py` junta as técnicas que compensaram: artigos/números, híbrida, reranking e expansão pelo grafo, com geração que cita o artigo exato.

```bash
cd modulos/06-rag-avancado/use-case
uv run python legislacao.py indexar
uv run python legislacao.py perguntar "Posso recusar fazer horas extra?"
uv run python legislacao.py avaliar
```

No módulo 7 vamos avaliar a qualidade das respostas deste assistente de forma sistemática.